Preparations

In [5]:
!pip install transformers==4.44.0 tokenizers==0.19.1 huggingface-hub==0.24.0 --quiet # because of issue with tokenisers in newer version
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import pandas as pd
import torch
import os
from google.colab import files
from google.colab import userdata
from huggingface_hub import login
from google.colab import drive
drive.mount("/content/drive")

login(userdata.get("hugging_face"))

# Setting save directory in Google Drive

save_dir = "/content/drive/MyDrive/tfm_results/news_dataset/"
os.makedirs(save_dir, exist_ok=True)

# Reading in dataframe

url = "https://raw.githubusercontent.com/bethancunningham/tfm/main/datasets/expanded_news_dataset.csv"
# For expanded version. For simple version: "https://raw.githubusercontent.com/bethancunningham/tfm/main/datasets/initial_news_dataset.csv"
df = pd.read_csv(url)

# Models

models = ["goldfish-models/cym_latn_5mb",
          "goldfish-models/cym_latn_10mb",
          "goldfish-models/cym_latn_100mb",
          "goldfish-models/cym_latn_1000mb",
          "britllm/britllm-3b-v0.1",
          "meta-llama/Llama-3.1-8B",
          "ai-forever/mGPT-13B",
          "microsoft/phi-2",
          "mistralai/Mistral-7B-v0.1",
          "bigscience/bloom-7b1",
          "britllm/TransWebLLM",
          "CohereLabs/tiny-aya-base",
          "utter-project/EuroLLM-9B",
          "Qwen/Qwen3-8B",
          "LLaMAX/LLaMAX3-8B"]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


Creating function for putting sentence pairs to the model and getting mean negative log likelihood across tokens for both sentences

NOTE: THE SECOND PART OF THIS FUNCTION WAS MY ORIGINAL ATTEMPT AT IMPLEMENTING THE LOGIT LENS FOR ALL MODELS BUT I DIDN'T END UP USING IT IT BECAUSE IT WAS MISSING THE LAYERNORM PART. I've kept it here because the results cleaning scripts expect the layer_nll columns, but I don't use them. The logit lens analysis only examines results from the Treebank dataset.

In [6]:
def calculate_NLL_final_and_layers(model, tokeniser, sentence_1, sentence_2): # sentence_1 is correct sentence, sentence_2 incorrect
    """Function to get NLL for correct sentence and incorrect sentence (mean across tokens) according to causal language model, as well as NLLs for each sentence at each layer.
    Taking the mean due to differing sentence lengths in terms of tokens in some pairs.
    Lower NLL indicates higher probability.
    Returns final NLL and NLLs at each layer for each sentence."""

    try :

      # Tokenising sentences

      inputs_1 = tokeniser(sentence_1, return_tensors="pt").to("cuda")
      inputs_2 = tokeniser(sentence_2, return_tensors="pt").to("cuda")

      with torch.no_grad(): # Just inference, not updating weights
          outputs_1 = model(**inputs_1, output_hidden_states=True) # Output hidden states to get logits
          outputs_2 = model(**inputs_2, output_hidden_states=True)

      tokens_1 = inputs_1["input_ids"][0] # 0 gives first (and only) sentence in batch as tokens - gets rid of batch dimension
      tokens_2 = inputs_2["input_ids"][0]

      # Calculating NLL across sentences

      log_probs_1 = torch.log_softmax(outputs_1.logits[0], dim=-1) # Calculating log probabilities of all possible next tokens after each token. dim=-1 is vocab size - so applying softmax to vocab
      log_probs_2 = torch.log_softmax(outputs_2.logits[0], dim=-1)
      nll_1_sum = -sum(log_probs_1[i, tokens_1[i+1].item()].item() for i in range(len(tokens_1)-1)) # At position i (certain token), what log prob was assigned to i+1 (next token)? Add together and - for nll
      nll_2_sum = -sum(log_probs_2[i, tokens_2[i+1].item()].item() for i in range(len(tokens_2)-1)) # -1 because no prediction carried out on last token
      nll_1_mean = nll_1_sum/(len(tokens_1)-1)
      nll_2_mean = nll_2_sum/(len(tokens_2)-1)

      # minus (log P(t1 | t0) + log P(t2 | t0,t1) + log P(t3 | t0,t1,t2) + ...)

      # Calculating sentence-level log prob at each layer

      lm_head = model.lm_head # Turns hidden states into logits
      layer_nll_correct = []
      layer_nll_incorrect = []

      for hidden_1, hidden_2 in zip(outputs_1.hidden_states, outputs_2.hidden_states) :

          # Projecting hidden states at each layer through lm_head - getting predictions at each layer

          logits_1_layer = lm_head(hidden_1[0])  # hidden_1[0] is first sequence in batch (only one sequence/sentence). Gives: (seq_len, vocab_size)
          logits_2_layer = lm_head(hidden_2[0])

          log_probs_1_layer = torch.log_softmax(logits_1_layer, dim=-1) # Calculating log probabilities of all possible next tokens after each token AT EACH LAYER. dim=-1 is vocab size
          log_probs_2_layer = torch.log_softmax(logits_2_layer, dim=-1) # Result is shape (seq_len, vocab size)

          """Note for Nora: Isn't this the same as logit lens? Only that, instead of taking the token with the highest probability for the next token, we're looking up the probability for the actual next token
          """

          # Summing log probs across all token positions (sentence-level NLL)
          nll_1_layer_sum = -sum(log_probs_1_layer[i, tokens_1[i+1].item()].item() for i in range(len(tokens_1)-1)) # Getting log probability of token i+1 given token i for all tokens in sentence, then sum and make negative
          nll_2_layer_sum = -sum(log_probs_2_layer[i, tokens_2[i+1].item()].item() for i in range(len(tokens_2)-1))
          nll_1_layer_mean = nll_1_layer_sum/(len(tokens_1)-1)
          nll_2_layer_mean = nll_2_layer_sum/(len(tokens_2)-1)

          layer_nll_correct.append(nll_1_layer_mean) # Creating list of NLLs at each layer
          layer_nll_incorrect.append(nll_2_layer_mean)

      return nll_1_mean, nll_2_mean, layer_nll_correct, layer_nll_incorrect

    except Exception as e :
      print(f"Error calculating NLL for pair: \n'{sentence_1}', \n'{sentence_2}: \n{e}")
      return float('inf'), float('inf'), float('inf'), float('inf') # Returning infinity for errors

Iterating over models and running NLL calculation function

In [7]:
for model_name in models:

    print(f'...processing {model_name}')

    # Loading model and tokeniser

    config = AutoConfig.from_pretrained(model_name)
    if not hasattr(config, 'pad_token_id') or config.pad_token_id is None: # dealing with issue with Phi 2 - no pad_token
        config.pad_token_id = config.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(model_name, config=config).to("cuda")
    model.tie_weights()
    tokeniser = AutoTokenizer.from_pretrained(model_name)

    # Creating list for results per row

    outs = []
    model_safe_name = model_name.replace("/", "_")

    # Iterating over rows in df, getting NLLs (mean across sentence, final layer and at each layer) and adding to df

    for i, row in tqdm(df.iterrows(), total=len(df)) : # tqdm notebook for progress bar
        correct_nll_mean, incorrect_nll_mean, layer_nll_correct, layer_nll_incorrect = calculate_NLL_final_and_layers(model, tokeniser, row.correct_sentence, row.incorrect_sentence)
        new_row = {
            'index': i,
            'model': model_name,
            'item': row.item,
            'sentence_id' : row.sent_id,
            'correct_form': row.correct_form,
            'sentence': row.correct_sentence,
            'incorrect_form': row.incorrect_form,
            'incorrect_sentence': row.incorrect_sentence,
            'mutation_type': row.mutation_type,
            'trigger_type': row.trigger_type,
            'specific_trigger': row.specific_trigger,
            'incorrect_form_mut_type' : row.incorrect_form_mut_type,
            'correct_nll_mean' : correct_nll_mean,
            'incorrect_nll_mean' : incorrect_nll_mean,
            'delta': incorrect_nll_mean - correct_nll_mean,
            'layer_nll_correct': layer_nll_correct,
            'layer_nll_incorrect': layer_nll_incorrect

        }
        outs.append(new_row)

        # Saving and downloading every 100 rows

        if (i + 1) % 100 == 0:
          checkpoint_path = f'{save_dir}/checkpoint_initial_news_dataset_{model_safe_name}_{i + 1}.csv' # For expanded dataset: checkpoint_expanded_news_dataset
          pd.DataFrame(outs).to_csv(checkpoint_path, index=False)
          print(f'... ... checkpoint saved at row {i + 1}')

    # Saving final results for model in Google Drive and printing accuracy

    if outs:
      final_path = f'{save_dir}/final_initial_news_dataset_{model_safe_name}_{i + 1}.csv' # For expanded dataset: final_expanded_news_dataset
      df_out = pd.DataFrame(outs)
      df_out.to_csv(final_path, index=False)

      accuracy = (df_out["delta"] > 0).mean()
      print(f"-------\n... {model_name} overall accuracy: {accuracy:.2%}")

      print(f"... accuracy by mutation_type:")
      print(df_out.groupby("mutation_type")["delta"].apply(lambda x: (x > 0).mean()).map("{:.2%}".format).to_string())

      print(f"... accuracy by trigger_type:")
      print(df_out.groupby("trigger_type")["delta"].apply(lambda x: (x > 0).mean()).map("{:.2%}".format).to_string())
      print("-------")
      del model
      torch.cuda.empty_cache()


...processing goldfish-models/cym_latn_5mb


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  0%|          | 0/503 [00:00<?, ?it/s]

... ... checkpoint saved at row 100
... ... checkpoint saved at row 200
... ... checkpoint saved at row 300
... ... checkpoint saved at row 400
... ... checkpoint saved at row 500
-------
... goldfish-models/cym_latn_5mb overall accuracy: 83.90%
... accuracy by mutation_type:
mutation_type
AM       71.88%
NM       86.67%
SM       85.28%
no_AM    96.97%
no_NM    88.46%
no_SM    85.19%
... accuracy by trigger_type:
trigger_type
L              82.08%
MS             83.33%
no_mutation    90.70%
-------
...processing goldfish-models/cym_latn_10mb


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/157M [00:00<?, ?B/s]

KeyboardInterrupt: 